In [1]:
import pandas as pd
import numpy as np

import datetime
import os, sys
import importlib

import utils
importlib.reload(utils)

from utils import plot_series, plot_series_with_names, plot_series_bar
from utils import plot_dataframe
from utils import get_universe_adjusted_series, scale_weights_to_one, scale_to_book_long_short
from utils import generate_portfolio, backtest_portfolio
from utils import match_implementations

import plotly.graph_objects as go

In [2]:
# This directory can be used if you're working on a Kaggle Notebook inside the competition
# Change the directory as per your requirements if you're working somewhere else
data_dir = "/kaggle/input/qrt-quant-quest-iit-bombay-2025/"

features = pd.read_parquet( "./features.parquet")

universe = pd.read_parquet("./universe.parquet")
 
returns = pd.read_parquet( "./returns.parquet")

defining the functions of training XG model after building the features 

In [5]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from scipy.stats import spearmanr

# Global variable to cache model for iterative portfolio generation
GLOBAL_MODEL = None
TRAIN_FEATURE_NAMES = None


def safe_div(a, b, eps=1e-8):
    return a / (b + eps)


def make_feature_dict(features: pd.DataFrame) -> dict:
    """
    features.parquet expected format:
    columns are MultiIndex: (feature_name, stock_id)
    index is dates
    """
    if not isinstance(features.columns, pd.MultiIndex):
        raise ValueError("Expected MultiIndex columns in features: (feature_name, stock_id)")
    feat_names = features.columns.get_level_values(0).unique()
    return {f: features[f].copy() for f in feat_names}


def build_engineered_features(feat: dict) -> dict:
    """Add engineered features while preserving time-causality."""
    eng = dict(feat)

    for k in [
        "macd", "trix", "relative_strength_index", "chaikin_money_flow",
        "trend_1_3", "trend_5_20", "trend_20_60", "volatility_20", "volatility_60"
    ]:
        if k in feat:
            eng[f"{k}_lag1"] = feat[k].shift(1)
            eng[f"{k}_delta_5"] = feat[k].shift(1) - feat[k].shift(6)

    if "trend_5_20" in feat and "volatility_20" in feat:
        eng["trend5_20_over_vol20"] = safe_div(
            feat["trend_5_20"].shift(1), feat["volatility_20"].shift(1)
        )

    if "trend_1_3" in feat and "trend_20_60" in feat:
        eng["trend_spread"] = feat["trend_1_3"].shift(1) - feat["trend_20_60"].shift(1)

    if "relative_strength_index" in feat:
        rsi = feat["relative_strength_index"].shift(1)
        eng["rsi_oversold"] = (rsi < 30).astype(float)
        eng["rsi_overbought"] = (rsi > 70).astype(float)

    return eng


def prepare_ranked_feature_panels(features: pd.DataFrame, universe: pd.DataFrame):
    """
    Precompute ranked feature panels once, reused for train/val builds.
    This removes repeated heavy DataFrame ranking work.
    """
    feat = make_feature_dict(features)
    eng = build_engineered_features(feat)

    feat_names = list(eng.keys())
    ranked_panels = {}
    tradable_mask = universe == 1

    for name in feat_names:
        masked = eng[name].where(tradable_mask)
        ranked = masked.rank(axis=1, pct=True, method="average")
        ranked_panels[name] = ranked.astype(np.float32)

    return feat_names, ranked_panels


def prepare_target_next_day_rank(returns: pd.DataFrame, universe: pd.DataFrame):
    """
    Build next-day cross-sectional return rank target.
    Target at row t corresponds to return rank at t+1.
    """
    target = returns.where(universe == 1).rank(axis=1, pct=True, method="average")
    return target.shift(-1).astype(np.float32)


def get_split_dates(universe: pd.DataFrame, start_date: str, end_date: str):
    dates = universe.loc[start_date:end_date].index
    if len(dates) < 2:
        raise ValueError(f"Need at least 2 trading days in split {start_date} to {end_date}")
    return dates


def build_matrix_from_cached_panels(
    ranked_panels: dict,
    target_next_rank: pd.DataFrame,
    universe: pd.DataFrame,
    feat_names: list,
    start_date: str,
    end_date: str,
):
    """
    Fast, split-safe matrix builder.
    Uses only rows inside [start_date, end_date], and drops split-end sample
    to avoid target crossing into another split.
    """
    split_dates = get_split_dates(universe, start_date, end_date)
    sample_dates = split_dates[:-1]  # enforce in-split t+1 target availability

    y_flat = target_next_rank.loc[sample_dates].to_numpy(dtype=np.float32, copy=False).reshape(-1)

    X_cols = []
    for name in feat_names:
        col = ranked_panels[name].loc[sample_dates].to_numpy(dtype=np.float32, copy=False).reshape(-1)
        X_cols.append(col)

    X_mat = np.column_stack(X_cols)
    valid_mask = np.isfinite(y_flat) & np.isfinite(X_mat).all(axis=1)

    X = X_mat[valid_mask]
    y = y_flat[valid_mask]
    meta = []

    return X, y, feat_names, meta


def train_xgb_ranker(X_train, y_train, X_val=None, y_val=None):
    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=5.0,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )

    if X_val is not None and y_val is not None:
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    else:
        model.fit(X_train, y_train, verbose=False)

    return model


def daily_ic_fast(
    model,
    ranked_panels: dict,
    target_next_rank: pd.DataFrame,
    universe: pd.DataFrame,
    feat_names: list,
    start_date: str,
    end_date: str,
):
    """
    Evaluate daily IC using the same split-safe timing contract as training.
    """
    split_dates = get_split_dates(universe, start_date, end_date)
    sample_dates = split_dates[:-1]

    ics = []
    for d in sample_dates:
        tradable = universe.loc[d] == 1
        tradable_idx = tradable[tradable].index

        if len(tradable_idx) < 10:
            continue

        X_day = np.column_stack([
            ranked_panels[name].loc[d, tradable_idx].to_numpy(dtype=np.float32, copy=False)
            for name in feat_names
        ])

        y_day = target_next_rank.loc[d, tradable_idx].to_numpy(dtype=np.float32, copy=False)
        mask = np.isfinite(y_day) & np.isfinite(X_day).all(axis=1)

        if mask.sum() < 10:
            continue

        preds = model.predict(X_day[mask])
        ic = spearmanr(preds, y_day[mask]).correlation
        if np.isfinite(ic):
            ics.append(ic)

    return {
        "mean_ic": float(np.mean(ics)) if len(ics) else np.nan,
        "std_ic": float(np.std(ics)) if len(ics) else np.nan,
        "num_days": int(len(ics)),
    }


def get_weights(features: pd.DataFrame, today_universe: pd.Series) -> dict[str, float]:
    """
    Intended inference function for generate_portfolio:
    - uses only historical features passed in
    - loads model from memory cache if available
    - enforces feature order used in training
    """
    global GLOBAL_MODEL, TRAIN_FEATURE_NAMES

    if GLOBAL_MODEL is None:
        if "model" not in globals():
            raise ValueError("Model is not trained yet. Run training cell first.")
        GLOBAL_MODEL = model

    tradable_stocks = today_universe[today_universe == 1].index
    if features.shape[0] < 6:
        return {str(s): 0.0 for s in tradable_stocks}

    df_today = features.iloc[-1].unstack(level=0).reindex(tradable_stocks)
    df_lag5 = features.iloc[-6].unstack(level=0).reindex(tradable_stocks)

    eng_feats = pd.DataFrame(index=tradable_stocks)
    base_cols = [
        "macd", "trix", "relative_strength_index", "chaikin_money_flow",
        "trend_1_3", "trend_5_20", "trend_20_60", "volatility_20", "volatility_60",
    ]

    for k in base_cols:
        if k in df_today.columns:
            eng_feats[f"{k}_lag1"] = df_today[k]
            eng_feats[f"{k}_delta_5"] = df_today[k] - df_lag5[k]

    if "trend_5_20" in df_today.columns and "volatility_20" in df_today.columns:
        eng_feats["trend5_20_over_vol20"] = df_today["trend_5_20"] / (df_today["volatility_20"] + 1e-8)

    if "trend_1_3" in df_today.columns and "trend_20_60" in df_today.columns:
        eng_feats["trend_spread"] = df_today["trend_1_3"] - df_today["trend_20_60"]

    if "relative_strength_index" in df_today.columns:
        eng_feats["rsi_oversold"] = (df_today["relative_strength_index"] < 30).astype(float)
        eng_feats["rsi_overbought"] = (df_today["relative_strength_index"] > 70).astype(float)

    X_today = eng_feats.rank(pct=True, method="average")

    if TRAIN_FEATURE_NAMES is not None:
        X_today = X_today.reindex(columns=TRAIN_FEATURE_NAMES)

    X_today = X_today.fillna(0.5).to_numpy(dtype=np.float32)

    preds = GLOBAL_MODEL.predict(X_today)
    scores_series = pd.Series(preds, index=tradable_stocks)

    ranked_scores = scores_series.rank(pct=True, method="average")
    centered = ranked_scores - ranked_scores.mean()
    denom = centered.abs().sum()

    if denom > 0:
        weights = centered / denom
    else:
        weights = pd.Series(0.0, index=tradable_stocks)

    weights.index = weights.index.astype(str)
    weights = weights.replace(0, np.nan).dropna()

    return weights.to_dict()

Executing the functions

In [6]:
import time

print("Status: Formatting datetime indices...")
t0 = time.time()
features.index = pd.to_datetime(features.index)
universe.index = pd.to_datetime(universe.index)
returns.index = pd.to_datetime(returns.index)
print(f"[✓] Indices formatted in {time.time() - t0:.2f} seconds.\n")

# Strict non-overlapping temporal splits
train_start, train_end = "2005-01-03", "2016-12-30"
val_start, val_end = "2017-01-02", "2019-12-31"
test_start, test_end = "2020-01-01", "2025-02-07"

train_dates = universe.loc[train_start:train_end].index
val_dates = universe.loc[val_start:val_end].index
test_dates = universe.loc[test_start:test_end].index

assert len(train_dates) > 1 and len(val_dates) > 1 and len(test_dates) > 0, "One split is empty or too short"
assert train_dates.max() < val_dates.min(), "Train/Validation order violation"
assert val_dates.max() < test_dates.min(), "Validation/Test order violation"
assert len(train_dates.intersection(val_dates)) == 0, "Train/Validation overlap detected"
assert len(train_dates.intersection(test_dates)) == 0, "Train/Test overlap detected"
assert len(val_dates.intersection(test_dates)) == 0, "Validation/Test overlap detected"

print("[✓] Split integrity checks passed")
print(f"    Train: {train_dates.min().date()} -> {train_dates.max().date()} ({len(train_dates)} days)")
print(f"    Valid: {val_dates.min().date()} -> {val_dates.max().date()} ({len(val_dates)} days)")
print(f"    Test : {test_dates.min().date()} -> {test_dates.max().date()} ({len(test_dates)} days)\n")

print("Status: Precomputing ranked feature panels and next-day targets...")
t_cache = time.time()
feat_names, ranked_panels = prepare_ranked_feature_panels(features, universe)
target_next_rank = prepare_target_next_day_rank(returns, universe)
print(f"[✓] Cache built in {time.time() - t_cache:.2f} seconds.\n")

print(f"Status: Building Training Data ({train_start} to {train_end})...")
t1 = time.time()
X_train, y_train, feat_names, _ = build_matrix_from_cached_panels(
    ranked_panels, target_next_rank, universe, feat_names, train_start, train_end
)
print(f"[✓] Training matrix built in {time.time() - t1:.2f} seconds.")
print(f"    Train shape: {X_train.shape}, {y_train.shape}\n")

print(f"Status: Building Validation Data ({val_start} to {val_end})...")
t2 = time.time()
X_val, y_val, _, _ = build_matrix_from_cached_panels(
    ranked_panels, target_next_rank, universe, feat_names, val_start, val_end
)
print(f"[✓] Validation matrix built in {time.time() - t2:.2f} seconds.")
print(f"    Val shape: {X_val.shape}, {y_val.shape}\n")

# Ensure inference uses exact training feature order
global TRAIN_FEATURE_NAMES
TRAIN_FEATURE_NAMES = feat_names

print("Status: Training XGBoost Ranker...")
t3 = time.time()
model = train_xgb_ranker(X_train, y_train, X_val, y_val)
print(f"[✓] Model training complete in {time.time() - t3:.2f} seconds.\n")

# Keep cached model for iterative get_weights
GLOBAL_MODEL = model

print("Status: Calculating Daily Information Coefficient (IC)...")
t4 = time.time()
metrics = daily_ic_fast(model, ranked_panels, target_next_rank, universe, feat_names, val_start, val_end)
print(f"[✓] IC evaluation complete in {time.time() - t4:.2f} seconds.\n")

print("=====================================")
print("Validation IC Stats:")
print("=====================================")
print(metrics)

Status: Formatting datetime indices...
[✓] Indices formatted in 0.16 seconds.

[✓] Split integrity checks passed
    Train: 2005-01-03 -> 2016-12-30 (3021 days)
    Valid: 2017-01-03 -> 2019-12-31 (754 days)
    Test : 2020-01-02 -> 2025-02-07 (1283 days)

Status: Precomputing ranked feature panels and next-day targets...
[✓] Cache built in 46.87 seconds.

Status: Building Training Data (2005-01-03 to 2016-12-30)...
[✓] Training matrix built in 8.37 seconds.
    Train shape: (2934633, 44), (2934633,)

Status: Building Validation Data (2017-01-02 to 2019-12-31)...
[✓] Validation matrix built in 1.10 seconds.
    Val shape: (746262, 44), (746262,)

Status: Training XGBoost Ranker...
[✓] Model training complete in 45.72 seconds.

Status: Calculating Daily Information Coefficient (IC)...
[✓] IC evaluation complete in 4.99 seconds.

Validation IC Stats:
{'mean_ic': 0.0033962115624381994, 'std_ic': 0.10553706767837476, 'num_days': 753}


In [7]:
import time

print("Status: Generating portfolio iteratively (this may take a few minutes)...")
start_time = time.time()

# 1. Generate the portfolio using your XGBoost get_weights function
# We use a 1-year validation period to test it quickly before running the full 20-year span.
xgb_portfolio = generate_portfolio(
    get_weights,
    features,
    universe,
    "2019-01-01",
    "2019-03-31",
)

generation_time = time.time() - start_time
print(f"[✓] Portfolio generated in {generation_time:.2f} seconds.")

print("Status: Running backtest...")




Status: Generating portfolio iteratively (this may take a few minutes)...


100%|██████████| 61/61 [01:17<00:00,  1.27s/it]


[✓] Portfolio generated in 78.01 seconds.
Status: Running backtest...


/Users/sambhavsinghaditya/Downloads/comet/qrt-quant-challenge-2026-iit-delhi/utils.py:301: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  portfolio = portfolio.fillna(0)


In [8]:
# 2. Backtest the generated portfolio using the daily returns
# The backtest_portfolio function takes (portfolio_weights, daily_returns, universe, plot_flag, verbose_flag)
sr, pnl = backtest_portfolio(
    xgb_portfolio.loc["2019-01-01":"2019-03-31"], 
    returns.loc["2019-01-01":"2019-03-31"], 
    universe.loc["2019-01-01":"2019-03-31"], 
    True,   # Set to True to plot the PnL curve
    True    # Set to True to print detailed stats
)

print("=====================================")
print("Backtest Results (2019-01-01 to 2019-03-31):")
print("=====================================")
print(f"Annualized Sharpe Ratio: {sr:.4f}")

/Users/sambhavsinghaditya/Downloads/comet/qrt-quant-challenge-2026-iit-delhi/utils.py:390: RuntimeWarning: invalid value encountered in scalar divide
  turnover = (traded.mean() / book_value.mean()) * 100
/Users/sambhavsinghaditya/Downloads/comet/qrt-quant-challenge-2026-iit-delhi/utils.py:394: RuntimeWarning: invalid value encountered in scalar divide
  gross_sharpe_ratio = (gross_pnl.mean() / gross_pnl.std()) * np.sqrt(252)
/Users/sambhavsinghaditya/Downloads/comet/qrt-quant-challenge-2026-iit-delhi/utils.py:396: RuntimeWarning: invalid value encountered in scalar divide
  net_sharpe_ratio = (net_pnl.mean() / net_pnl.std()) * np.sqrt(252)


Gross Sharpe Ratio:  nan
Net Sharpe Ratio:  nan
Turnover %:  nan


Backtest Results (2019-01-01 to 2019-03-31):
Annualized Sharpe Ratio: nan


In [ ]:
universe["2005-01-03":"2005-01-10"]

,1,2,3,4,5,6,7,8,9,10,...,2158,2159,2160,2161,2162,2163,2164,2165,2166,2167
Date,,,,,,,,,,,,,,,,,,,,,
2005-01-03,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2005-01-04,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2005-01-05,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2005-01-06,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2005-01-07,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2005-01-10,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
